# Phase 3b: a latent traffic regime, Event ← hour, weekday (Issue #19)

**Question.** Does a hidden regime ("Event") explain why the gases rise and fall together, and do its learned states line up with recognisable traffic patterns, such as weekday rush hours?

**Network** (Figure 3b, without the AQ node):

```
hour, weekday (or day of week)  →  Event (latent, K states)
Event (+ hour, in the ablation) →  CO, C6H6, NOx, NO2
gas  →  nominal sensor (solid), cross-sensitive sensors (dashed, from #16)
W = T + AH  →  every sensor
```

AQ is not part of the network. It is Phase 3a's rule applied to the estimated concentrations.

**Assumptions** (report §3)
- **B1.** Given Event, the gases are independent of each other, and independent of hour (Event-only version).
- **B2.** Event depends only on hour and day type.
- **B3.** Hours are independent given the CPTs. Event isn't chained across hours; that is Phase 4.
- Plus Phase 2's sensor assumptions, and missing-at-random given the inputs.

**What this notebook covers**
1. Setup, with the sensor edges from #16
2. A vectorized BN for the grid (LayeredBN)
3. The grid: K ∈ {2, 3, 4} × {Event-only, Event + hour} × {is_weekday, dow} × 5 EM restarts × 3 split seeds
4. Choosing K: validation log-likelihood, vs Phase 2, and stability across restarts
5. EM convergence
6. What the Event states mean: P(gas | Event), P(Event | hour, weekday) and the posterior by hour
7. The B1 check: do the gases stop co-moving once Event is known?
8. The "some event" at K = 4: a seasonal regime, and why we don't keep it
9. Summary for the report

The per-gas metrics, concentration MAE and the AQ comparison against Phase 2 and the HourOnlyBaseline are in the partner's part of #19. Section 10 gives them the fitted models.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import itertools
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.optimize import linear_sum_assignment

from cse250a.modules.config import config
from cse250a.modules.config.paths import REPO_ROOT
from cse250a.modules.core import TIME_CARDS, Gas, Latent, Network, Sensor, Time, Weather
from cse250a.modules.data import Discretizer, blocked_split, load_air_quality
from cse250a.modules.eval import bin_means
from cse250a.modules.models.layered import LayeredBN

FIG = "phase3b"
GRID_CSV = REPO_ROOT / "notebooks" / "experiments" / "phase3b_grid.csv"
STAB_CSV = REPO_ROOT / "notebooks" / "experiments" / "phase3b_stability.csv"
RERUN_GRID = (
    False  # True re-fits all 180 + 3 models (~15–30 min); False loads the saved CSVs
)

SPLIT_SEEDS = (0, 1, 2)
RESTART_SEEDS = (0, 1, 2, 3, 4)
KS = (2, 3, 4)
EVENT = Latent.EMISSION_REGIME  # the latent "Event" node

plt.rcParams.update(
    {"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False}
)

In [ ]:
df = load_air_quality(config.paths.air_quality_csv)
WEATHER = (Weather.T, Weather.AH)
GASES = (Gas.CO, Gas.C6H6, Gas.NOX, Gas.NO2)  # NMHC: ~90% missing, not modeled (#16)
SENSORS = tuple(Sensor)

before = len(df)
df = df.dropna(subset=[Time.HOUR, *WEATHER])
print(f"dropped {before - len(df)} rows with missing hour/weather; {len(df)} left")

splits = {}
for seed in SPLIT_SEEDS:
    train_raw, val_raw, _ = blocked_split(df, seed=seed)  # test is not touched here
    disc = Discretizer(n_bins=config.bins.n_bins).fit(
        train_raw, [*SENSORS, *GASES, *WEATHER]
    )
    splits[seed] = {
        "train_raw": train_raw,
        "val_raw": val_raw,
        "train": disc.transform(train_raw),
        "val": disc.transform(val_raw),
        "cards": {**disc.cards, **TIME_CARDS},
    }
pd.DataFrame(
    {s: {"train": len(v["train"]), "val": len(v["val"])} for s, v in splits.items()}
).rename_axis("seed", axis=1)

**Sensor edges** are the final dicts from the cross-sensitivity study (#16, `notebooks/phase2/cross_sensitivity.ipynb`). They are the same in every model here, so differences between models come only from the Event part.

In [ ]:
nominal = {Sensor.S1: (Gas.CO,), Sensor.S3: (Gas.NOX,), Sensor.S4: (Gas.NO2,)}
cross = {
    Sensor.S2: (Gas.C6H6,),
    Sensor.S4: (Gas.CO, Gas.NOX),
    Sensor.S5: (Gas.CO, Gas.NOX),
}
SENSOR_PARENTS = {
    s: (*nominal.get(s, ()), *cross.get(s, ()), *WEATHER) for s in SENSORS
}


def event_network(K, parents, day, cards):
    """parents: "event" (Event-only, hour → Event → gas) or "event+hour" (hour → gas kept)."""
    gas_parents = (EVENT,) if parents == "event" else (EVENT, Time.HOUR)
    return Network(
        structure={
            EVENT: (Time.HOUR, day),
            **{g: gas_parents for g in GASES},
            **SENSOR_PARENTS,
        },
        cards={**cards, EVENT: K},
        latent=(EVENT,),
    )


def phase2_network(cards):
    """Phase 2 on the same sensors: hour → gas, gases independent given hour (A1)."""
    return Network(
        structure={**{g: (Time.HOUR,) for g in GASES}, **SENSOR_PARENTS}, cards=cards
    )


def free_params(net):
    return int(
        sum(np.prod(net.shape(n)[:-1]) * (net.shape(n)[-1] - 1) for n in net.structure)
    )


cards0 = splits[0]["cards"]
print(event_network(3, "event", Time.IS_WEEKDAY, cards0))

## 2. A vectorized BN for the grid

`DiscreteBN.fit` loops over rows in Python. On this network one EM iteration takes ~3 s, and the grid needs 180 fits, which would be 15–30 hours. `LayeredBN` (`src/cse250a/modules/models/layered.py`) is a subclass that keeps DiscreteBN's `fit`, M-step, initialization, MAP objective and `predict_proba`. It only replaces the E-step, `log_likelihood` and `posterior` with numpy over all rows at once. It enumerates the K × 3⁴ joint states of (Event, gases) per row, so it is still exact.

`tests/test_layered.py` checks it against DiscreteBN on real rows with missing gases and sensors, for the Event-only, Event + hour and Phase 2 structures. It compares the EM objective at every iteration, every CPT, the expected counts, the log-likelihood, the posteriors and `predict_proba`, and they agree to ~1e-10. A full fit takes ~3–20 s.

## 3. The grid

For each (K, parents, day, split seed):
- Fit 5 EM restarts, with `max_iter=200` and the project's default `tol`.
- Keep the restart with the best **train** objective, and record every restart's objective.
- Score the kept model on **val**: log-likelihood of the observed cells per hour, conditional on the inputs (hour, day, T, AH).

Phase 2 is fitted on the same splits, bins and sensor edges for comparison.

**Stability.** For every (K, parents, seed) we also compare each restart's P(gas | Event) with the best restart's, after matching up the state labels (Hungarian algorithm). A restart "agrees" if no probability differs by 0.1 or more.

With `RERUN_GRID = False` the cell loads the saved results from `notebooks/experiments/`.

In [ ]:
def gas_profile(bn):
    """P(gas | Event [, hour]) stacked into one (K, ...) array, for matching states across restarts."""
    return np.concatenate(
        [bn.cpts[g].table.reshape(bn.network.cards[EVENT], -1) for g in GASES], axis=1
    )


def agreement(profiles):
    """Max |ΔP| of each restart vs the first (best) one, after aligning state labels."""
    ref, out = profiles[0], []
    for p in profiles[1:]:
        cost = np.abs(ref[:, None, :] - p[None, :, :]).sum(axis=2)
        _, col = linear_sum_assignment(cost)
        out.append(np.abs(ref - p[col]).max())
    return out


if RERUN_GRID or not GRID_CSV.exists():
    rows, stab, t0 = [], [], time.time()
    for seed in SPLIT_SEEDS:
        sp = splits[seed]
        bn = LayeredBN(
            phase2_network(sp["cards"]), alpha=config.bins.alpha, max_iter=200
        ).fit(sp["train"])
        rows.append(
            {
                "K": 0,
                "parents": "phase2",
                "day": "-",
                "seed": seed,
                "restart": 0,
                "train_obj": bn.result.final,
                "n_iter": bn.result.n_iter,
                "converged": bn.result.converged,
                "monotone": bn.result.is_monotone(),
                "val_ll": bn.log_likelihood(sp["val"]) / len(sp["val"]),
                "params": free_params(bn.network),
            }
        )
        for K, parents, day in itertools.product(
            KS, ("event", "event+hour"), (Time.IS_WEEKDAY, Time.DOW)
        ):
            fits = []
            for r in RESTART_SEEDS:
                net = event_network(K, parents, day, sp["cards"])
                bn = LayeredBN(net, alpha=config.bins.alpha, max_iter=200, seed=r).fit(
                    sp["train"]
                )
                fits.append(bn)
                rows.append(
                    {
                        "K": K,
                        "parents": parents,
                        "day": str(day),
                        "seed": seed,
                        "restart": r,
                        "train_obj": bn.result.final,
                        "n_iter": bn.result.n_iter,
                        "converged": bn.result.converged,
                        "monotone": bn.result.is_monotone(),
                        "val_ll": bn.log_likelihood(sp["val"]) / len(sp["val"]),
                        "params": free_params(net),
                    }
                )
            fits.sort(key=lambda b: -b.result.final)
            d = agreement([gas_profile(b) for b in fits])
            stab.append(
                {
                    "K": K,
                    "parents": parents,
                    "day": str(day),
                    "seed": seed,
                    "restarts_agreeing": int(sum(x < 0.1 for x in d)),
                    "max_dP": max(d),
                }
            )
            print(
                f"seed {seed}  K={K}  {parents:10s}  {day:10s}  done ({time.time() - t0:.0f}s)"
            )
    grid = pd.DataFrame(rows)
    stability = pd.DataFrame(stab)
    GRID_CSV.parent.mkdir(parents=True, exist_ok=True)
    grid.to_csv(GRID_CSV, index=False)
    stability.to_csv(STAB_CSV, index=False)
else:
    grid, stability = pd.read_csv(GRID_CSV), pd.read_csv(STAB_CSV)

print(
    f"{len(grid)} fits | all EM runs monotone: {grid['monotone'].all()} | converged: {grid['converged'].mean():.0%} | "
    f"iterations: median {grid['n_iter'].median():.0f}, max {grid['n_iter'].max()}"
)

**Explanation:** every EM run's objective (log-likelihood + log-prior) increases at every iteration, which is the check that the E- and M-steps are correct, and every run converges well before the 200-iteration cap.

## 4. Choosing K

In [ ]:
# best restart (by train objective) per (K, parents, day, seed)
event = grid[grid["parents"] != "phase2"]
best = event.loc[event.groupby(["K", "parents", "day", "seed"])["train_obj"].idxmax()]
phase2 = grid[grid["parents"] == "phase2"].set_index("seed")["val_ll"]

summary = best.groupby(["K", "parents", "day"]).agg(
    val_ll=("val_ll", "mean"), val_ll_std=("val_ll", "std"), params=("params", "first")
)
gain = best.assign(vs_phase2=best["val_ll"] - best["seed"].map(phase2))
summary["vs Phase 2"] = gain.groupby(["K", "parents", "day"])["vs_phase2"].mean()
summary["restart spread (train obj)"] = (
    event.groupby(["K", "parents", "day", "seed"])["train_obj"]
    .agg(lambda x: x.max() - x.min())
    .groupby(["K", "parents", "day"])
    .mean()
)
stab_sum = stability.groupby(["K", "parents", "day"]).agg(
    restarts_agreeing=("restarts_agreeing", "mean"), max_dP=("max_dP", "max")
)
summary = summary.join(stab_sum)
summary["stable"] = summary["max_dP"] < 0.1
print(
    f"Phase 2 val log-lik per hour: {phase2.mean():.4f} ± {phase2.std():.4f}  "
    f"({grid.loc[grid['parents'] == 'phase2', 'params'].iloc[0]} params)"
)
summary.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, day in zip(axes, ("is_weekday", "dow")):
    for parents, color in (("event", "#1565C0"), ("event+hour", "#EF6C00")):
        s = summary.xs((parents, day), level=("parents", "day"))
        ax.errorbar(
            s.index,
            s["val_ll"],
            yerr=s["val_ll_std"],
            marker="o",
            capsize=3,
            color=color,
            label=f"{'Event-only' if parents == 'event' else 'Event + hour'}",
        )
        for K, row in s.iterrows():
            if not row["stable"]:
                ax.scatter(
                    K,
                    row["val_ll"],
                    s=160,
                    facecolors="none",
                    edgecolors="#C62828",
                    lw=1.5,
                )
    ax.axhline(phase2.mean(), color="grey", ls="--", label="Phase 2 (hour → gas)")
    ax.set_xticks(KS)
    ax.set_xlabel("K (Event states)")
    ax.set_title(f"day variable: {day}")
axes[0].set_ylabel("val log-likelihood per hour (nats)")
axes[1].scatter(
    [],
    [],
    s=160,
    facecolors="none",
    edgecolors="#C62828",
    label="unstable across restarts",
)
axes[1].legend(loc="lower right", fontsize=8)
fig.suptitle("Model selection: mean ± std over split seeds 0–2 (higher is better)")
plt.tight_layout()
plt.savefig(config.paths.figure(FIG, "model_selection.png"), bbox_inches="tight")
plt.show()

In [ ]:
# paired differences (same split seed), the fair way to compare configurations
piv = best.set_index(["K", "parents", "day", "seed"])["val_ll"].unstack("seed")


def paired(a, b):
    d = piv.loc[a] - piv.loc[b]
    return pd.Series(
        {
            "mean Δ": d.mean(),
            "std Δ": d.std(),
            "per seed": np.round(d.values, 3).tolist(),
        }
    )


comparisons = {
    "Event-only: K=3 vs K=2": paired(
        (3, "event", "is_weekday"), (2, "event", "is_weekday")
    ),
    "Event-only: K=4 vs K=3": paired(
        (4, "event", "is_weekday"), (3, "event", "is_weekday")
    ),
    "K=3: Event + hour vs Event-only": paired(
        (3, "event+hour", "is_weekday"), (3, "event", "is_weekday")
    ),
    "K=4: Event + hour vs Event-only": paired(
        (4, "event+hour", "is_weekday"), (4, "event", "is_weekday")
    ),
    "K=3 Event-only: dow vs is_weekday": paired(
        (3, "event", "dow"), (3, "event", "is_weekday")
    ),
    "K=3 Event-only vs Phase 2": pd.Series(
        {
            "mean Δ": (piv.loc[(3, "event", "is_weekday")] - phase2).mean(),
            "std Δ": (piv.loc[(3, "event", "is_weekday")] - phase2).std(),
            "per seed": np.round(
                (piv.loc[(3, "event", "is_weekday")] - phase2).values, 3
            ).tolist(),
        }
    ),
}
pd.DataFrame(comparisons).T

**Explanation:**
- **Every Event model fits far better than Phase 2.** K = 3 Event-only gains about **+0.62 nats per hour** on validation, and it does so with *fewer* parameters on the gas side. Phase 2 needs 24 × 2 free parameters per gas. Event-only needs K × 2 per gas plus one P(Event | hour, weekday) table. The gain comes from Phase 2's assumption A1 (gases independent given hour) being badly wrong: knowing one gas tells you a lot about the others, and Event carries exactly that shared information.
- **Validation log-likelihood alone keeps rising with K.** K = 3 vs 2 is +0.29, and K = 4 vs 3 is +0.07 per hour, both positive in every seed. Event + hour fits slightly better than Event-only (+0.04 at K = 3).
- **But stability rules most of those out.** The stability columns show that Event-only K = 2 and K = 3 reach the *same* solution from every restart (largest difference ≈ 0.03–0.05 with is_weekday). Event-only K = 4 does not: restarts disagree by up to 0.74, so the 4th state means something different each time. And **Event + hour is never reproducible** (differences ≈ 0.95 for every K). When hour also feeds the gases directly, many different Event solutions explain the data equally well, so the states carry no stable meaning.
- **Day variable.** Day of week adds +0.005 per hour over weekday/weekend at K = 3. That is consistent in sign but negligible, while the Event table is 3.5× larger.

**Selection rule:** highest validation log-likelihood among configurations whose EM solution is reproducible across restarts (all 5 restarts agree within 0.1, in every split seed). When two configurations are within 0.01 nats per hour, we take the smaller one.

→ **Chosen: K = 3, Event-only, is_weekday.** All test-set reporting uses this configuration once.

In [ ]:
CHOSEN = {"K": 3, "parents": "event", "day": Time.IS_WEEKDAY}


def fit_chosen(seed, K=CHOSEN["K"], parents=CHOSEN["parents"], day=CHOSEN["day"]):
    """Re-fit the best restart of a configuration on one split (the grid only saved scores)."""
    rows = best[
        (best["K"] == K)
        & (best["parents"] == parents)
        & (best["day"] == str(day))
        & (best["seed"] == seed)
    ]
    r = int(rows["restart"].iloc[0])
    sp = splits[seed]
    net = event_network(K, parents, day, sp["cards"])
    return LayeredBN(net, alpha=config.bins.alpha, max_iter=200, seed=r).fit(
        sp["train"]
    )


chosen = {seed: fit_chosen(seed) for seed in SPLIT_SEEDS}

## 5. EM convergence (chosen configuration, all restarts, seed 0)

In [ ]:
sp = splits[0]
fig, ax = plt.subplots(figsize=(7, 3.5))
for r in RESTART_SEEDS:
    net = event_network(CHOSEN["K"], CHOSEN["parents"], CHOSEN["day"], sp["cards"])
    bn = LayeredBN(net, alpha=config.bins.alpha, max_iter=200, seed=r).fit(sp["train"])
    h = bn.result.to_frame()["objective"]
    ax.plot(
        h.index + 1, h.values, marker=".", label=f"restart {r} (final {h.iloc[-1]:.0f})"
    )
ax.set_xlabel("EM iteration")
ax.set_ylabel("train log-lik + log-prior")
ax.set_title("EM objective, K = 3 Event-only (seed 0): monotone, restarts end together")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(config.paths.figure(FIG, "em_convergence.png"), bbox_inches="tight")
plt.show()

## 6. What the Event states mean

In [ ]:
def state_order(bn):
    """Sort states from cleanest to most polluted (expected CO bin), so labels match across seeds."""
    return np.argsort(bn.cpts[Gas.CO].table @ np.arange(bn.network.cards[Gas.CO]))


NAMES = {0: "quiet (night / weekend)", 1: "moderate (daytime)", 2: "rush hour"}

rows = []
for seed, bn in chosen.items():
    order = state_order(bn)
    sp = splits[seed]
    for g in GASES:
        table = bn.cpts[g].table[order]  # (K, bins)
        c_bar = bin_means(sp["train_raw"][g], sp["train"][g])
        for k in range(CHOSEN["K"]):
            rows.append(
                {
                    "seed": seed,
                    "state": NAMES[k],
                    "gas": str(g).split("(")[0],
                    **{f"P(bin {b})": table[k, b] for b in range(table.shape[1])},
                    "E[concentration]": table[k] @ c_bar.to_numpy(),
                }
            )
p_gas = pd.DataFrame(rows)
p_gas_mean = (
    p_gas.groupby(["state", "gas"], sort=False)
    .mean(numeric_only=True)
    .drop(columns="seed")
)
p_gas_mean.round(3)

**P(gas | Event)**, averaged over the three split seeds, with states sorted by CO level. `E[concentration]` is Σ P(bin) · (bin mean from train): mg/m³ for CO, µg/m³ for C6H6 and NO2, ppb for NOx.

In [ ]:
fig, axes = plt.subplots(
    CHOSEN["K"], 1, figsize=(10, 1.6 * CHOSEN["K"] + 0.8), sharex=True
)
for k, ax in enumerate(axes):
    avg = np.mean(
        [
            chosen[s].cpts[EVENT].table[:, :, state_order(chosen[s])[k]]
            for s in SPLIT_SEEDS
        ],
        axis=0,
    )
    sns.heatmap(
        avg.T[::-1],
        ax=ax,
        vmin=0,
        vmax=1,
        cmap="magma",
        cbar=k == 0,
        yticklabels=["weekday", "weekend"],
        xticklabels=range(24),
        cbar_kws={"label": "P(state | hour, day)"} if k == 0 else None,
    )
    ax.set_ylabel(NAMES[k], rotation=0, ha="right", va="center")
axes[-1].set_xlabel("hour of day")
fig.suptitle("P(Event | hour, weekday): learned prior, mean of split seeds 0–2")
plt.tight_layout()
plt.savefig(
    config.paths.figure(FIG, "p_event_given_hour_weekday.png"), bbox_inches="tight"
)
plt.show()

In [ ]:
# Posterior P(Event | e) on validation, e = what's visible at predict time (no gas references)
evidence = (Time.HOUR, Time.IS_WEEKDAY, *WEATHER, *SENSORS)
frames = []
for seed, bn in chosen.items():
    val = splits[seed]["val"]
    post = bn.predict_proba(val, EVENT, evidence).to_numpy()[:, state_order(bn)]
    f = pd.DataFrame(
        post, columns=[NAMES[k] for k in range(CHOSEN["K"])], index=val.index
    )
    f["hour"], f["day"] = (
        val[Time.HOUR].values,
        np.where(val[Time.IS_WEEKDAY] == 1, "weekday", "weekend"),
    )
    frames.append(f)
post_val = pd.concat(frames)
by_hour = post_val.groupby(["day", "hour"]).mean()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
colors = ["#2E7D32", "#F9A825", "#C62828"]
for ax, day in zip(axes, ("weekday", "weekend")):
    by_hour.loc[day].plot.area(
        ax=ax, color=colors, alpha=0.85, legend=False, linewidth=0
    )
    ax.set_title(day)
    ax.set_xlabel("hour of day")
    ax.set_xlim(0, 23)
    ax.set_ylim(0, 1)
axes[0].set_ylabel("mean P(Event | evidence)")
axes[1].legend(loc="upper left", bbox_to_anchor=(1, 1), fontsize=8)
fig.suptitle(
    "Posterior over Event on validation, averaged by hour (evidence: hour, day, weather, sensors)"
)
plt.tight_layout()
plt.savefig(config.paths.figure(FIG, "posterior_by_hour.png"), bbox_inches="tight")
plt.show()

In [ ]:
# how consistent are the states across split seeds? (after sorting by CO)
prof = {
    s: np.stack([chosen[s].cpts[g].table[state_order(chosen[s])] for g in GASES])
    for s in SPLIT_SEEDS
}
print(
    "max |ΔP(gas | Event)| between split seeds:",
    round(
        max(
            np.abs(prof[a] - prof[b]).max()
            for a, b in itertools.combinations(SPLIT_SEEDS, 2)
        ),
        3,
    ),
)
print(
    "share of hours (val posterior):",
    post_val[[NAMES[k] for k in range(CHOSEN["K"])]].mean().round(3).to_dict(),
)

**Explanation:** the three states have a clear, reproducible meaning. Every restart reaches the same solution, and the three split seeds differ by at most ≈ 0.1 in any P(gas | Event):

| State | Gases | When (learned prior) |
|---|---|---|
| **quiet** | All four gases almost always in their lowest bin (CO ≈ 0.8 mg/m³) | Early morning (~02:00–06:00) on every day, and part of weekend daytime |
| **moderate** | Mostly the middle bin (CO ≈ 1.7 mg/m³) | Weekday midday and late evening (22:00–00:00), and weekend evenings |
| **rush hour** | All four gases mostly in their top bin (CO ≈ 3.5 mg/m³, NOx ≈ 360 ppb) | **Weekday 07:00–10:00 and 17:00–21:00.** It largely disappears on weekends, except a weaker evening peak around 19:00. |

This is exactly the traffic pattern the question asked about. The two weekday peaks match commuter traffic, and the weekend drop matches lower traffic. The posterior on validation is computed from hour, day, weather and sensors only, with no reference readings. It shows the same shape, so the sensors can recognise the regime on hours the model never saw.

## 7. The B1 check: do the gases stop co-moving once Event is known?

Phase 2 assumed A1: gases are independent given hour. B1 says they are independent given Event. We measure the average rank correlation between the 6 gas pairs on validation hours where all four gases are observed, in four ways:

- **raw**: no conditioning
- **within hour**: Spearman within each hour of day, averaged (what A1 claims is 0)
- **within Event**: posterior-weighted rank correlation inside each Event state, averaged by state size. The posterior uses all observed cells (gases and sensors).
- **within Event (sensors only)**: the same, but Event is inferred without the gas readings. This is a stricter version, because the posterior can't simply sort hours by the gases it is being tested on.

In [ ]:
def weighted_corr(x, y, w):
    mx, my = np.average(x, weights=w), np.average(y, weights=w)
    return np.sum(w * (x - mx) * (y - my)) / np.sqrt(
        np.sum(w * (x - mx) ** 2) * np.sum(w * (y - my) ** 2)
    )


def within_event(full, post, a, b):
    ra, rb = full[a].rank().to_numpy(), full[b].rank().to_numpy()
    return np.average(
        [weighted_corr(ra, rb, post[:, k]) for k in range(post.shape[1])],
        weights=post.sum(axis=0),
    )


import warnings

from scipy.stats import ConstantInputWarning

warnings.filterwarnings(
    "ignore", category=ConstantInputWarning
)  # an hour with one constant gas

b1 = []
for seed, bn in chosen.items():
    full = splits[seed]["val"].dropna(subset=list(GASES))
    post_all = bn.posterior(full, EVENT)
    post_sensors = bn.posterior(full.assign(**{g: np.nan for g in GASES}), EVENT)
    for a, b in itertools.combinations(GASES, 2):
        b1.append(
            {
                "seed": seed,
                "pair": f"{a.split('(')[0]}–{b.split('(')[0]}",
                "n": len(full),
                "raw": full[a].corr(full[b], method="spearman"),
                "within hour": np.nanmean(
                    [
                        g[a].corr(g[b], method="spearman")
                        for _, g in full.groupby(Time.HOUR)
                    ]
                ),
                "within Event": within_event(full, post_all, a, b),
                "within Event (sensors only)": within_event(full, post_sensors, a, b),
            }
        )
b1 = pd.DataFrame(b1)
b1_table = b1.groupby("pair").mean(numeric_only=True).drop(columns="seed")
b1_table.loc["mean of pairs"] = b1_table.mean()
b1_table.round(3)

In [ ]:
cols = ["raw", "within hour", "within Event (sensors only)", "within Event"]
fig, ax = plt.subplots(figsize=(8, 3.6))
b1_table.drop(index="mean of pairs")[cols].plot.bar(
    ax=ax, color=["#9E9E9E", "#EF6C00", "#64B5F6", "#1565C0"], width=0.8
)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("rank correlation between gases")
ax.set_xlabel("")
ax.set_title(
    "B1 check (validation, mean of seeds 0–2): conditioning on Event removes most co-movement"
)
ax.legend(fontsize=8)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(config.paths.figure(FIG, "b1_check.png"), bbox_inches="tight")
plt.show()

**Explanation:**
- **A1 fails, as expected.** Within a single hour of day, the gases are still strongly correlated (≈ 0.52 on average, against ≈ 0.65 raw). Knowing the hour removes only about a fifth of the co-movement.
- **Event removes most of it.** Inside an Event state the remaining correlation is close to zero (≈ 0.07). Even when Event is inferred from the sensors alone, so it can't "cheat" with the gas readings, it drops to ≈ 0.25, about half the within-hour value.
- **B1 is not perfectly true**, and the report should say so. NOx–NO2 keeps a correlation of ≈ 0.46 inside Event, which makes sense because NO2 is part of NOx. CO–C6H6 also keeps a little (≈ 0.15), since both come straight from exhaust. But B1 is a much better approximation than A1, and this is where the +0.6 nats per hour gain over Phase 2 comes from.

## 8. The "some event" at K = 4: a seasonal regime, and why we don't keep it

Validation log-likelihood still improves from K = 3 to K = 4 (+0.07 per hour). So what is the 4th state? Below we look at the **best** K = 4 Event-only solution for seed 0, and where its states fall over the calendar.

In [ ]:
bn4 = fit_chosen(0, K=4)
order4 = state_order(bn4)
profile4 = pd.DataFrame(
    {str(g).split("(")[0]: bn4.cpts[g].table[order4] @ np.arange(3) for g in GASES},
    index=[f"state {i}" for i in range(4)],
).round(2)
print("expected bin (0 = low, 2 = high) per state, sorted by CO:")
print(profile4)

train = splits[0]["train"]
post4 = pd.DataFrame(
    bn4.posterior(train, EVENT)[:, order4],
    index=train.index,
    columns=[f"state {i}" for i in range(4)],
)
by_month = post4.groupby(train.index.to_period("M")).mean()

fig, ax = plt.subplots(figsize=(9, 3.4))
by_month.index = by_month.index.strftime("%b %y")
by_month.plot.bar(
    stacked=True, ax=ax, color=["#2E7D32", "#81C784", "#FFB300", "#C62828"], width=0.85
)
ax.set_ylabel("share of hours (posterior)")
ax.set_xlabel("")
ax.set_title("K = 4 Event-only (seed 0, best restart): share of each state by month")
ax.legend(bbox_to_anchor=(1, 1), fontsize=8)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(config.paths.figure(FIG, "k4_states_by_month.png"), bbox_inches="tight")
plt.show()

**Explanation:** at K = 4 the "moderate" state splits in two, and the split is **not about the hour**. It is a **gas-mix regime that follows the season**:
- One of the two middle states has CO and C6H6 at medium levels but **low NOx and NO2**. It covers ≈ 40% of hours in August (likely the *Ferragosto* holidays, when Italian city traffic drops) and ~20% in spring and summer, but almost none from November to March.
- The other middle state has **higher NOx and NO2**, and it grows through autumn and winter, to ≈ 40% of hours in January–March 2005. That fits winter conditions: heating emissions, and cold, stable air that traps NOx near the ground.

So the extra "event" the model finds is seasonal, not a traffic pattern. Phase 4 (HMM) or a month/season input would be the right way to model it.

**Why we don't keep K = 4:** its solution isn't reproducible. Across the 5 restarts the 4th state lands somewhere different (P(gas | Event) differs by up to 0.74 after aligning labels), and in seed 0 the other restarts end 29–189 nats below the best. The seasonal story above is the best restart's story, but another run of EM would tell a different one. That is the label-switching / local-optima problem the issue anticipated. Reporting K = 3, with this section as the discussion of what more states pick up, is the honest choice.

## 9. Summary for the report

**Model and choice.** A latent Event with K = 3 states, parents (hour, weekday), and the four gases as its children (Event-only), with the sensor edges from #16.
- K was chosen on validation log-likelihood among configurations whose EM solution is reproducible across 5 restarts in all 3 split seeds.
- K = 4 and every Event + hour model fit slightly better, but they fail that test.
- Day of week adds nothing over weekday/weekend (+0.005 nats per hour for a 3.5× larger table).

**Results**

| Item | Result |
|---|---|
| Fit | +0.62 nats per hour on validation over Phase 2 (same rows, same evidence), with fewer gas-side parameters. Every EM run is monotone and converges in < 40 iterations. |
| States | quiet (early morning / weekend), moderate (daytime), and rush hour (weekday 07–10 and 17–21 h). Every restart agrees, and seeds differ by ≤ 0.1. |
| B1 | Mean gas–gas correlation: ≈ 0.65 raw, ≈ 0.52 within hour (A1 fails), ≈ 0.07 within Event, and ≈ 0.25 when Event is inferred from sensors only. NOx–NO2 keeps ≈ 0.46. |
| "Some event" | At K = 4 the extra state is a seasonal gas-mix regime: low NOx in summer/August, high NOx in winter. It isn't reproducible, so it is discussed, not kept. |

**Limitations**
- EM finds local optima, so we use restarts and keep the best train objective.
- State labels are arbitrary, so we sort them by CO level and match them with the Hungarian algorithm.
- Event can't capture slow, multi-day changes (season, holidays), because it is re-drawn independently every hour (B3). That is Phase 4's job.
- B1 still leaves NOx–NO2 dependence, since NO2 is part of NOx.
- Event + hour fits best, but its states aren't identifiable.

**Decisions** (recorded in `docs/decisions.md`): K = 3, Event-only, is_weekday.

## 10. For the comparison part of #19

The partner's part covers per-gas metrics, concentration MAE, AQ precision/recall/F1 against Phase 2 and the HourOnlyBaseline, and the test set. It can start from `chosen[seed]` (fitted `LayeredBN`s, one per split seed), using the same evidence as above:

```python
evidence = (Time.HOUR, Time.IS_WEEKDAY, *WEATHER, *SENSORS)
proba = chosen[seed].predict_proba(splits[seed]["val"], Gas.CO, evidence)
est = expected_concentration(proba, bin_means(splits[seed]["train_raw"][Gas.CO], splits[seed]["train"][Gas.CO]))
```

`LayeredBN` is a `DiscreteBN`, so `predict_proba`, `evaluate` and `log_likelihood` work as usual, only faster. Phase 2 can be fitted the same way with `LayeredBN(phase2_network(cards))`.

*AI use: analysis drafted with help from Claude, reviewed and run by me.*

## 10. Comparisons

Imports needed for the comparison part

### Part 1: Set-up for Comparisons

- Import concentration, HourOnlyBaseline

In [ ]:
from cse250a.modules.eval import (
    concentration_error,
    expected_concentration,
    resolution_floor,
)
from cse250a.modules.models import HourOnlyBaseline

- Taking information from phase 3-a

In [ ]:
evidence = (Time.HOUR, Time.IS_WEEKDAY, *WEATHER, *SENSORS)

OFFICIAL_THRESHOLDS = {Gas.CO: 10.0, Gas.C6H6: 5.0, Gas.NOX: 16.0, Gas.NO2: 200.0}
RELATIVE_QUANTILE = 0.90
CHOSEN_K = 2   # Phase 3a's k; confirm against its printed CHOSEN_K

print("bins:", config.bins.n_bins)   # should be 3, matching Phase 3a

In [ ]:
def thresholds_for(train_raw):
    return {
        "official": OFFICIAL_THRESHOLDS.copy(),
        "relative": {g: float(train_raw[g].quantile(RELATIVE_QUANTILE)) for g in GASES},
    }

- Fitting the models

In [ ]:
models = {}   # models[seed] = {"event": ..., "phase2": ..., "hour": ...}
for seed in SPLIT_SEEDS:
    sp = splits[seed]
    models[seed] = {
        "event": chosen[seed],
        "phase2": LayeredBN(
            phase2_network(sp["cards"]), alpha=config.bins.alpha, max_iter=200
        ).fit(sp["train"]),
        "hour": HourOnlyBaseline(cards=sp["cards"], alpha=config.bins.alpha).fit(sp["train"]),
    }
print({s: list(m) for s, m in models.items()})

In [ ]:
sp = splits[0]
n = len(sp["val"])
print("event val ll/hr:", models[0]["event"].log_likelihood(sp["val"]) / n)
print("grid value:     ", grid[(grid.K == 3) & (grid.parents == "event")
      & (grid.day == "is_weekday") & (grid.seed == 0)]["val_ll"].max())

for name, m in models[0].items():
    p = m.predict_proba(sp["val"], Gas.CO, evidence)
    print(name, p.shape, p.sum(axis=1).round(3).unique())

*AI use: code started with help from Claude, edited, reviewed and run by me.*